In [1]:
import io
import json
import os
from PIL import Image, ImageDraw, ImageFont
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()  # This loads the variables from your .env file into os.environ

def get_scalable_font(font_name_options, font_size):
    """Attempts to load a scalable font from a list of standard system font names."""
    for font_name in font_name_options:
        try:
            return ImageFont.truetype(font_name, size=font_size)
        except IOError:
            continue
    return ImageFont.load_default()

def draw_wrapped_text(draw, text, xmin, ymin, xmax, font, fill_color, line_spacing=4):
    """Helper function to wrap long effects text so it fits within the horizontal boundary."""
    words = text.split(' ')
    lines = []
    current_line = []
    
    for word in words:
        if '\n' in word:
            parts = word.split('\n')
            current_line.append(parts[0])
            lines.append(' '.join(current_line))
            for part in parts[1:-1]:
                lines.append(part)
            current_line = [parts[-1]] if parts[-1] else []
        else:
            current_line.append(word)
            test_line = ' '.join(current_line)
            left, top, right, bottom = font.getbbox(test_line)
            line_width = right - left
            max_width = xmax - xmin - 16  # Include padding margins
            
            if line_width > max_width:
                current_line.pop()
                lines.append(' '.join(current_line))
                current_line = [word]
                
    if current_line:
        lines.append(' '.join(current_line))
        
    current_y = ymin + 8
    for line in lines:
        draw.text((xmin + 8, current_y), line, fill=fill_color, font=font)
        left, top, right, bottom = font.getbbox(line if line else "A")
        line_height = bottom - top
        current_y += line_height + line_spacing

In [2]:
# Initialize the Google GenAI Client
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [ ]:
input_path = "input.png"
output_path = "output.png"

# Open the initial image
orig_img = Image.open(input_path).convert("RGB")
orig_width, orig_height = orig_img.size

In [ ]:
# --- PASS 1: UNIVERSAL DYNAMIC TEXT EXTRACTION ---
print("Extracting English translation blocks contextually...")
extraction_prompt = (
    "Look at the English text written directly below the card layout in this image. "
    "Extract the text components cleanly and map them into a JSON object with these keys:\n"
    "1. 'name': The main identity string / character title.\n"
    "2. 'type': The faction/archetype class type text (e.g., <insert list of example types>).\n"
    "3. 'effects': All the combined rules paragraphs containing activation requirements and conditions.\n"
    "Return the response exclusively as valid raw JSON without markdown markers."
)

txt_response = client.models.generate_content(
    model='gemini-3.5-flash-lite',#'gemini-3.5-flash','gemini-3.6-flash',
    contents=[orig_img, extraction_prompt],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
    ),
)

try:
    english_translations = json.loads(txt_response.text)
    print("Successfully extracted English translations layout:")
    print(json.dumps(english_translations, indent=2))
except json.JSONDecodeError:
    print("Failed to dynamically extract translation strings from the image template.")
    raise

In [ ]:
print('english_translations',english_translations)

In [ ]:
# --- PASS 2: COGNITIVE GEOMETRIC LAYOUT SEGMENTATION ---
# The Japanese text instructions are completely removed below.
# The script now looks purely for layout hierarchy blocks.
prompt = (
    "Analyze this image containing a trading card. Detect the precise bounding boxes "
    "[ymin, xmin, ymax, xmax] normalized from 0 to 1000 based strictly on visual zones:\n"
    "1. 'card_boundary': The outer perimeter enclosing the physical borders of the trading card.\n"
    "2. 'effects': The main rules text box located near the middle-to-bottom section of the card artwork.\n"
    "3. 'name': The primary prominent name line banner banner located at the bottom-center of the card, immediately above the smaller type line.\n"
    "4. 'type': The tiny descriptive classification field directly centered beneath the primary character name.\n"
    "Return the response exclusively as a valid JSON dictionary containing these keys.",
    "dictionary needs to look like the following: ",
    "{",
    " 'card_boundary': ... ",
    " 'effects': ... ",
    " 'name': ...",
    " 'type': ..."
    "}"
)

print("Requesting universal layout structure analysis from Gemini...")
response = client.models.generate_content(
    model='gemini-3.6-flash',#'gemini-3-flash-preview',#'gemini-3.5-flash',#'gemini-3.6-flash',
    contents=[orig_img, prompt],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
    ),
)

In [ ]:
print('response', response)

In [12]:
try:
    bboxes = json.loads(response.text)
except json.JSONDecodeError:
    print("Failed to parse JSON layout from the model.")
    raise


In [ ]:
bboxes

In [ ]:
# 3. Crop to the card boundary first if found
if "card_boundary" in bboxes:
    card_box = bboxes["card_boundary"]
    c_ymin = int(card_box[0] * orig_height / 1000)
    c_xmin = int(card_box[1] * orig_width / 1000)
    c_ymax = int(card_box[2] * orig_height / 1000)
    c_xmax = int(card_box[3] * orig_width / 1000)
    
    img = orig_img.crop((c_xmin, c_ymin, c_xmax, c_ymax))
    print("Successfully cropped image to the card boundaries.")
else:
    print("Card boundary not detected. Proceeding with original image.")
    img = orig_img
    c_xmin, c_ymin = 0, 0

# Get updated dimensions of the processed image
width, height = img.size
draw = ImageDraw.Draw(img)

font_candidates = ["arial.ttf", "Helvetica.ttc", "LiberationSans-Regular.ttf"]

# 4. Modify the text fields
for field_key, box in bboxes.items():
    if field_key == "card_boundary" or field_key not in english_translations:
        continue

    try:
        orig_ymin = int(box[0] * orig_height / 1000)
        orig_xmin = int(box[1] * orig_width / 1000)
        orig_ymax = int(box[2] * orig_height / 1000)
        orig_xmax = int(box[3] * orig_width / 1000)
        
        ymin = orig_ymin - c_ymin if "card_boundary" in bboxes else orig_ymin
        xmin = orig_xmin - c_xmin if "card_boundary" in bboxes else orig_xmin
        ymax = orig_ymax - c_ymin if "card_boundary" in bboxes else orig_ymax
        xmax = orig_xmax - c_xmin if "card_boundary" in bboxes else orig_xmax
        
        ymin, xmin = max(0, ymin), max(0, xmin)
        ymax, xmax = min(height, ymax), min(width, xmax)
        
        box_height = ymax - ymin
        
        if field_key in ["name", "type"]:
            font_size = int(box_height * 0.55)
        else:
            font_size = int(box_height * 0.12)  # Slightly reduced scale to gracefully manage long text variants
            
        font = get_scalable_font(font_candidates, font_size)
        bg_pixel = img.getpixel((xmin + 2, ymin + 2))
        
        draw.rectangle([xmin, ymin, xmax, ymax], fill=bg_pixel)
        text_color = (255, 255, 255) if sum(bg_pixel) / 3 < 128 else (0, 0, 0)
        
        text_content = english_translations[field_key]
        
        if field_key == "effects":
            draw_wrapped_text(draw, text_content, xmin, ymin, xmax, font, text_color)
        else:
            draw.text((xmin + 8, ymin + 6), text_content, fill=text_color, font=font)
            
        print(f"Successfully processed field: {field_key} with font size {font_size}")
    except:
        continue

img.save(output_path)
print(f"Process finalized! Translated layout saved to: {output_path}")